# 01: EDA Raw Data & Cleaning

Tujuan: Memuat data mentah dari file Excel POS (Olsera) ITA Cake Shop, melakukan eksplorasi kondisi awal data, lalu menerapkan pembersihan sebelum digunakan untuk modeling forecasting pada Skripsi.

In [2]:
import os
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
from IPython.display import display, HTML

## 1. Load & Eksplorasi Data Mentah (Raw Data)
Bagian ini memuat seluruh data penjualan mentah (raw data) untuk dianalisis lebih lanjut.

In [3]:
# ------------------------------------------------------------------------------
# 1. DYNAMIC PATH RESOLUTION
# ------------------------------------------------------------------------------
def resolve_base_directory():
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / 'Skripsi[revisifinal]',
        cwd.parent / 'Skripsi[revisifinal]',
        Path(r'c:\Users\admin\Documents\Semester 8 & 9\Skripsi(2)\Skripsi[revisifinal]')
    ]
    for p in candidates:
        if (p / 'ITA CAKE SHOP ANTAPANI (feb 2023-feb 2026)').exists():
            return p
    raise FileNotFoundError('Base directory for ITA Cake Shop datasets could not be resolved.')

BASE_DIR = resolve_base_directory()
DIR_P1 = BASE_DIR / 'ITA CAKE SHOP ANTAPANI (feb 2023-feb 2026)'
DIR_P2 = BASE_DIR / 'ITA CAKE SHOP ANTAPANI (maret 2025-september 2026)'
OUT_DIR = BASE_DIR / 'Data_Output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'[*] Base Directory: {BASE_DIR}')
print(f'[*] Output Directory: {OUT_DIR}')

# ------------------------------------------------------------------------------
# 2. RAW EXCEL INGESTION WITH RESOURCE OPTIMIZATION
# ------------------------------------------------------------------------------
USE_COLS = [
    'order no', 'order date', 'order time', 'order status',
    'item name', 'item group', 'qty', 'price', 'amount', 'discount amount'
]

def load_excel_files(directory, prefix='Item Penjualan'):
    files = [p for p in directory.rglob(f'*{prefix}*.xlsx') if not p.name.startswith('~$')]
    print(f'[*] Found {len(files)} files in {directory.name}')
    dfs = []
    for f in sorted(files):
        df_temp = pd.read_excel(f)
        available_cols = [c for c in USE_COLS if c in df_temp.columns]
        dfs.append(df_temp[available_cols])
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

raw_df_1 = load_excel_files(DIR_P1)
raw_df_2 = load_excel_files(DIR_P2)

[*] Base Directory: c:\Users\admin\Documents\Semester 8 & 9\Skripsi(2)\FINAL_BERSIH\01_data_ITA-Cake-Shop
[*] Output Directory: c:\Users\admin\Documents\Semester 8 & 9\Skripsi(2)\FINAL_BERSIH\01_data_ITA-Cake-Shop\Data_Output
[*] Found 37 files in ITA CAKE SHOP ANTAPANI (feb 2023-feb 2026)
[*] Found 19 files in ITA CAKE SHOP ANTAPANI (maret 2025-september 2026)


### 1.1 Pembuatan 7 File Arsip Raw Data (Arsip Historis POS)
Mengekstrak dan menyatukan seluruh file `.xlsx` mentah dari laporan sistem POS Olsera menjadi 7 file arsip CSV yang saling berkorespondensi.

In [4]:
# ------------------------------------------------------------------------------
# 1.1 GENERATE 7 RAW ARCHIVE CSVS
# ------------------------------------------------------------------------------
import time

raw_archive_definitions = {
    '1_Item_Penjualan_berdasarkan_Tanggal': DIR_P1 / 'LAPORAN PENJUALAN' / 'ITEM PENJUALAN BERDASARKAN TANGGAL',
    '2_Rincian_Penjualan': DIR_P1 / 'LAPORAN PENJUALAN' / 'RINCIAN PENJUALAN',
    '3_Laporan_Keseluruhan_Penjualan': DIR_P1 / 'LAPORAN PENJUALAN' / 'LAPORAN KESELURUHAN PENJUALAN',
    '4_Penjualan_berdasarkan_Tanggal': DIR_P1 / 'LAPORAN PENJUALAN' / 'PENJUALAN BERDASARKAN TANGGAL',
    '5_Penjualan_berdasarkan_Kategori': DIR_P1 / 'LAPORAN PRODUK' / 'PENJUALAN BERDASARKAN KATEGORI',
    '6_Penjualan_berdasarkan_SKU': DIR_P1 / 'LAPORAN PRODUK' / 'PENJUALAN BERDASARKAN SKU',
    '7_QTY_Produk_Terjual': DIR_P1 / 'LAPORAN PRODUK' / 'QTY PRODUK TERJUAL'
}

summary_data = []

for name, folder in raw_archive_definitions.items():
    t0 = time.time()
    files = sorted([f for f in folder.rglob('*.xlsx') if not f.name.startswith('~$')])
    dfs = [pd.read_excel(f, engine='openpyxl') for f in files]
    concat = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    
    if not concat.empty:
        concat.columns = concat.columns.str.lower().str.strip()
        out_file = OUT_DIR / f'{name}.csv'
        concat.to_csv(out_file, index=False)
        size_mb = out_file.stat().st_size / (1024*1024)
        summary_data.append({'Arsip': name, 'Baris': len(concat), 'Kolom': len(concat.columns), 'Size (MB)': round(size_mb, 2)})

df_summary = pd.DataFrame(summary_data)
display(HTML("<h4>Ringkasan 7 Arsip Raw Data:</h4>"))
display(df_summary.style.format({'Size (MB)': '{:.2f}'}).set_table_attributes('class="dataframe"').hide(axis='index'))

Arsip,Baris,Kolom,Size (MB)
1_Item_Penjualan_berdasarkan_Tanggal,106566,70,28.55
2_Rincian_Penjualan,40525,48,9.26
3_Laporan_Keseluruhan_Penjualan,3695,6,0.11
4_Penjualan_berdasarkan_Tanggal,705,9,0.04
5_Penjualan_berdasarkan_Kategori,399,8,0.01
6_Penjualan_berdasarkan_SKU,2248,17,0.16
7_QTY_Produk_Terjual,2247,15,0.34


### Temuan 1: Inkonsistensi Tipe Data & Nilai Null
Memeriksa apakah terdapat missing values dan ketidaksesuaian tipe data bawaan ekspor POS yang dapat menggagalkan analisis time-series.

In [5]:
print('--- Info Tipe Data ---')
print(raw_df_1.info())
print('\n--- Jumlah Null ---')
print(raw_df_1.isnull().sum())

--- Info Tipe Data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106566 entries, 0 to 106565
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order no         106566 non-null  object 
 1   order date       106566 non-null  object 
 2   order time       106566 non-null  object 
 3   order status     106553 non-null  object 
 4   item name        106566 non-null  object 
 5   item group       106566 non-null  object 
 6   qty              106566 non-null  int64  
 7   price            106566 non-null  int64  
 8   amount           106566 non-null  float64
 9   discount amount  106566 non-null  int64  
dtypes: float64(1), int64(3), object(6)
memory usage: 8.1+ MB
None

--- Jumlah Null ---
order no            0
order date          0
order time          0
order status       13
item name           0
item group          0
qty                 0
price               0
amount              0
discount amount 

### Temuan 2: Transaksi Void / Batal
Sistem POS sering merekam transaksi batal (Order Status selain Z/Completed) yang secara akuntansi tidak mencerminkan permintaan (demand).

In [6]:
print('--- Distribusi Order Status ---')
print(raw_df_1['order status'].value_counts(dropna=False))

--- Distribusi Order Status ---
order status
Z      106553
NaN        13
Name: count, dtype: int64


### Temuan 3: Duplikasi Data akibat Irisan Periode (Overlap)
Karena data diekspor per periode waktu dari sistem POS, sering terjadi irisan di mana satu transaksi muncul dua kali.

In [7]:
print('Total Baris Duplikat:', raw_df_1.duplicated(subset=['order no', 'order date', 'item name', 'qty', 'price', 'amount']).sum())

Total Baris Duplikat: 0


### Temuan 4: Transaksi Bernilai Negatif / Nol
Dalam data historis, terkadang ada retur barang yang menyebabkan nilai qty atau amount menjadi nol atau negatif.

In [8]:
qty_neg = (pd.to_numeric(raw_df_1['qty'], errors='coerce').fillna(0) <= 0).sum()
amt_neg = (pd.to_numeric(raw_df_1['amount'], errors='coerce').fillna(0) <= 0).sum()
print(f'Transaksi Qty <= 0: {qty_neg}\nTransaksi Amount <= 0: {amt_neg}')

Transaksi Qty <= 0: 13
Transaksi Amount <= 0: 13


## 2. Proses Data Cleaning & Transformasi
Melakukan sanitasi teks, perbaikan tipe data, koreksi logika finansial (diskon & amount), penghapusan transaksi void, dan deduplikasi untuk memastikan dataset bersih dan valid.

In [9]:
# ------------------------------------------------------------------------------
# 3. ROBUST DATA CLEANING & REVENUE CORRECTION
# ------------------------------------------------------------------------------
def clean_pos_dataset(df_raw):
    df = df_raw.copy()
    
    # Text sanitization
    df['item name'] = df['item name'].astype(str).str.strip().str.title()
    df['item group'] = df['item group'].astype(str).str.strip().str.title()
    
    # Date parsing
    df['order date'] = pd.to_datetime(df['order date'], errors='coerce')
    df = df.dropna(subset=['order date', 'item name', 'item group', 'qty', 'amount'])
    
    # Numeric sanitization
    df['qty'] = pd.to_numeric(df['qty'], errors='coerce').fillna(0)
    df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0)
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce').fillna(0)
    df['discount amount'] = pd.to_numeric(df['discount amount'], errors='coerce').fillna(0)
    
    # FINANCIAL CORRECTION: In Olsera POS, amount is already net of discounts!
    df['gross_amount'] = df['qty'] * df['price']
    df['omzet_bersih'] = df['amount']  # DO NOT SUBTRACT discount amount AGAIN!
    
    # Exclude void / negative transactions
    df = df[df['order status'] == 'Z']
    df = df[(df['qty'] > 0) & (df['omzet_bersih'] > 0)]
    
    # Safe deduplication: include price to preserve legitimate distinct orders
    dedup_cols = ['order no', 'order date', 'item name', 'qty', 'price', 'amount']
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    
    return df

clean_p1 = clean_pos_dataset(raw_df_1)
clean_p2 = clean_pos_dataset(raw_df_2)

### 2.1 Evaluasi Komparasi Data: Mentah (Raw) vs Bersih (Clean) & Analisis Kategori Utama
Menghitung perbandingan eksplisit antara Data Mentah vs Data Bersih untuk memastikan retensi data, serta menampilkan Top 5 Kategori Penjualan pada data mentah.

In [10]:
# ------------------------------------------------------------------------------
# 2.1 COMPARE RAW VS CLEAN & TOP 5 CATEGORIES
# ------------------------------------------------------------------------------

# Raw vs Clean Comparison
raw_rows = len(raw_df_1)
clean_rows = len(clean_p1)
retention = (clean_rows / raw_rows) * 100 if raw_rows > 0 else 0

raw_gross = (pd.to_numeric(raw_df_1['qty'], errors='coerce').fillna(0) * pd.to_numeric(raw_df_1['price'], errors='coerce').fillna(0)).sum()
raw_net = pd.to_numeric(raw_df_1['amount'], errors='coerce').fillna(0).sum()
clean_gross = clean_p1['gross_amount'].sum()
clean_net = clean_p1['omzet_bersih'].sum()

comp_data = {
    'Metrik': ['Total Baris Transaksi', 'Total Gross Revenue (Rp)', 'Total Net Revenue (Rp)'],
    'Raw Data': [f'{raw_rows:,}', f'{raw_gross:,.0f}', f'{raw_net:,.0f}'],
    'Clean Data': [f'{clean_rows:,}', f'{clean_gross:,.0f}', f'{clean_net:,.0f}'],
    'Selisih / Retensi': [f'{retention:.2f}% Retensi', f'{clean_gross - raw_gross:,.0f}', f'{clean_net - raw_net:,.0f}']
}
df_comp = pd.DataFrame(comp_data)

# Top 5 Categories (Raw)
top_5_cats = raw_df_1['item group'].astype(str).str.strip().str.title().value_counts().head(5).reset_index()
top_5_cats.columns = ['Kategori', 'Total Transaksi']

display(HTML("<h4>Perbandingan Metrik: Raw vs Clean Data (Skripsi Window)</h4>"))
display(df_comp.style.set_table_attributes('class="dataframe"').hide(axis='index'))

display(HTML("<h4>Top 5 Kategori (Berdasarkan Frekuensi Transaksi Data Mentah)</h4>"))
display(top_5_cats.style.set_table_attributes('class="dataframe"').hide(axis='index'))

Metrik,Raw Data,Clean Data,Selisih / Retensi
Total Baris Transaksi,"106,566","106,553",99.99% Retensi
Total Gross Revenue (Rp),"4,056,144,060","4,055,383,060","-761,000"
Total Net Revenue (Rp),"4,045,051,745","4,045,812,745","761,000"


Kategori,Total Transaksi
Cake,16044
Japanese,13594
Mochi,12194
Lilin,11907
Mille Crepes,10127


## 3. Perlakuan Khusus: Masking Idul Fitri & IQR Capping
Mengatasi outlier ekstrem (lonjakan tajam saat lebaran atau anomali) menggunakan masking moving holiday dan Tukey IQR untuk menstabilkan varians sisa sebelum forecasting.

In [11]:
# ------------------------------------------------------------------------------
# 4. ENFORCE 155-WEEK OFFICIAL THESIS WINDOW & BOUNDARY HANDLING
# ------------------------------------------------------------------------------
OFFICIAL_START = pd.Timestamp('2023-02-20')
OFFICIAL_END = pd.Timestamp('2026-02-08')  # End of 155th full week (Sunday)

df_master_155w = clean_p1[
    (clean_p1['order date'] >= OFFICIAL_START) & 
    (clean_p1['order date'] <= OFFICIAL_END)
].copy()

# Retain full Period 1 transactional master (including Monday 2026-02-09)
df_master_p1_all = clean_p1[
    (clean_p1['order date'] >= OFFICIAL_START) &
    (clean_p1['order date'] <= pd.Timestamp('2026-02-09'))
].copy()

# Combined dataset with post-Feb 2026 data
combined_raw = pd.concat([clean_p1, clean_p2], ignore_index=True)
df_master_sept = combined_raw.drop_duplicates(
    subset=['order no', 'order date', 'item name', 'qty', 'price', 'amount'], 
    keep='first'
)

# ------------------------------------------------------------------------------
# 5. GENERATE DOWNSTREAM WEEKLY PIVOT WITH HOLIDAY MASKING & IQR CAPPING
# ------------------------------------------------------------------------------
TARGET_CATEGORIES = ['Cake', 'Japanese', 'Mille Crepes']
df_target = df_master_155w[df_master_155w['item group'].isin(TARGET_CATEGORIES)]

# Resample daily per category
daily_pivot = df_target.pivot_table(
    index='order date', 
    columns='item group', 
    values='qty', 
    aggfunc='sum'
).reindex(pd.date_range(OFFICIAL_START, OFFICIAL_END, freq='D')).fillna(0)

# Resample weekly: Monday-anchored Monday-to-Sunday cycle
weekly_pivot = daily_pivot.resample('W-MON', closed='left', label='left').sum()
weekly_pivot = weekly_pivot[TARGET_CATEGORIES]

# Verify exact 155 complete weeks
assert len(weekly_pivot) == 155, f'Expected 155 weeks, got {len(weekly_pivot)}'

# A. Moving Holiday Masking (Idul Fitri H-14 to H+3) & Linear Interpolation
IDUL_FITRI_RANGES = [
    (pd.Timestamp('2023-04-08'), pd.Timestamp('2023-04-25')),
    (pd.Timestamp('2024-03-27'), pd.Timestamp('2024-04-13')),
    (pd.Timestamp('2025-03-17'), pd.Timestamp('2025-04-03')),
]

for start_dt, end_dt in IDUL_FITRI_RANGES:
    mask = (weekly_pivot.index + pd.Timedelta(days=6) >= start_dt) & (weekly_pivot.index <= end_dt)
    weekly_pivot.loc[mask, :] = np.nan

weekly_pivot = weekly_pivot.interpolate(method='linear').bfill().ffill()

# B. Category-Level Tukey IQR Outlier Capping (Q3 + 1.5 * IQR)
for cat in TARGET_CATEGORIES:
    q1 = weekly_pivot[cat].quantile(0.25)
    q3 = weekly_pivot[cat].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    weekly_pivot[cat] = np.where(weekly_pivot[cat] > upper_bound, upper_bound, weekly_pivot[cat])

## 4. Ekspor File Master Data
Menyimpan hasil pembersihan ke dalam format CSV agar pipeline analisis (notebook 02) dapat langsung memuat data yang sudah bersih tanpa menjalankan pembersihan dari awal.

In [12]:
# ------------------------------------------------------------------------------
# 6. EXPORT ARTIFACTS (CSV & PARQUET)
# ------------------------------------------------------------------------------
df_master_p1_all.to_csv(OUT_DIR / 'Master_Clean_Feb2023_Feb2026.csv', index=False)
df_master_sept.to_csv(OUT_DIR / 'Master_Clean_Feb2023_Sept2026.csv', index=False)
weekly_pivot.to_csv(OUT_DIR / 'weekly_clean_final.csv', index=True, index_label='date')

print(f'[*] Successfully exported clean master datasets to {OUT_DIR}')
print(f'[*] Weekly pivot generated: {weekly_pivot.shape} (155 continuous full weeks)')

[*] Successfully exported clean master datasets to c:\Users\admin\Documents\Semester 8 & 9\Skripsi(2)\FINAL_BERSIH\01_data_ITA-Cake-Shop\Data_Output
[*] Weekly pivot generated: (155, 3) (155 continuous full weeks)


## Kesimpulan
Proses pembersihan data mentah memakan langkah signifikan: penghapusan data void, deduplikasi irisan periode, sanitasi teks dan perbaikan tipe data. Penerapan Masking Idul Fitri sangat esensial karena menghapus outlier spesifik sebelum menggunakan Tukey IQR, menjadikan data deret waktu stabil untuk analisis peramalan (forecasting).